
# F2b/F3 corrected separability-scale audit

Aquest notebook refà la figura de separabilitat local amb `epsilon^2` de Kruskal--Wallis i bootstrap per caminades.

Correccions importants respecte el codi anterior:

1. **Estandardització fixa**: cada coordenada es transforma a z-score una sola vegada amb la mitjana i desviació estàndard de la mostra observada completa. Els bootstraps no reestandarditzen la coordenada.
2. **Finestres fixes**: els centres de les finestres es calculen una vegada en la mostra observada. Els bootstraps avaluen exactament la mateixa escala.
3. **KW tie-corrected**: `scipy.stats.kruskal` calcula `H` amb correcció per empats.
4. **`epsilon^2 >= 0`**: els valors negatius són soroll de mostra finita; per defecte es retallen a zero.
5. **Bootstrap auditable**: desa percentil, basic/pivotal, normal i centred. El plot usa per defecte `basic` perquè és el que vols revisar.
6. **Null per caminada opcional**: el null pot permutar globalment o dins de cada caminada. Per una afirmació local/within-walk és més conservador usar permutació dins de caminada.

Nota curta: un interval de confiança formal **no està obligat** a contenir l'estimador observat. Si vols una banda visual al voltant de la línia observada, usa `CI_MODE = "normal"` o `CI_MODE = "centered"` i anomena-ho “bootstrap error band”, no “percentile CI”.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Carpeta on tens aquest notebook i compute_f2b_scale_fixed.py
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

# Canvia això si cal.
DATA_PATH = Path("../TFM-Thermal-Walks/ORGANISED/effective_temperature_results_final/votes_with_Tenv_comfort.csv")
OUTDIR = Path("./f2b_fixed_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

COORDS = ["T_corr", "HDX_corr", "T_env"]
STATE_COL = "state3"   # canvia-ho si uses comfort3_option1 o un altre nom
WALK_COL = "walk"

DELTAS = np.round(np.arange(0.4, 3.01, 0.2), 2)
N_CENTRES = 14
MIN_WIN = 30
MIN_GROUP = 5

NBOOT = 1000      # per prova ràpida pots posar 200; per figura final 1000 o 2000
NPERM = 500       # per figura final 500 o 1000
LEVEL = 0.68      # equivalent aproximat a 1 sigma; posa 0.95 si vols banda 95%
CI_MODE = "basic" # "basic", "percentile", "normal", "centered"
AGGREGATOR = "median"  # "median" recomanat; alternatives: "mean", "weighted_mean"
PERMUTE_WITHIN_WALK = True
SEED = 0


In [ ]:

from compute_f2b_scale_fixed import (
    prepare_data, compute, infer_floor, plot_f2b, plot_f3_contrast,
    write_audit_report
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No trobo DATA_PATH={DATA_PATH}. Edita la cel·la de configuració amb la ruta correcta."
    )

prep = prepare_data(
    data_path=DATA_PATH,
    coords=COORDS,
    state_col=STATE_COL,
    walk_col=WALK_COL,
    deltas=DELTAS,
    n_centres=N_CENTRES,
    center_q_low=2.0,
    center_q_high=98.0,
)

print(f"Files/vots usats: {len(prep.df)}")
print(f"Caminades: {prep.df[WALK_COL].nunique()}")
print("Comptes de classes 0/1/2:")
display(prep.df["lab"].value_counts().sort_index().rename("n").to_frame())

print("Mitjanes i SD globals usades per estandarditzar:")
display(pd.DataFrame({c: {"mean": prep.df[c].mean(), "sd": prep.df[c].std(ddof=1)} for c in COORDS}).T)


In [ ]:

results = compute(
    prep=prep,
    nboot=NBOOT,
    nperm=NPERM,
    level=LEVEL,
    ci_mode=CI_MODE,
    min_win=MIN_WIN,
    min_group=MIN_GROUP,
    aggregator=AGGREGATOR,
    seed=SEED,
    permute_within_walk=PERMUTE_WITHIN_WALK,
    clip_zero=True,
    outdir=OUTDIR,
)

results_path = OUTDIR / "_f2b_data_fixed.csv"
results.to_csv(results_path, index=False)
print(f"Desat: {results_path}")
display(results.head(12))


In [ ]:
results = pd.read_csv("f2b_fixed_final/_f2b_data_fixed.csv")
# Diagnòstic de biaix i de si l'observat cau dins cada banda.
rows = []
for mode in ["percentile", "basic", "normal", "centered"]:
    ok = (results[f"lo_{mode}"] <= results["obs"]) & (results["obs"] <= results[f"hi_{mode}"])
    rows.append({"mode": mode, "obs_inside": int(ok.sum()), "total": len(ok)})

diag = pd.DataFrame(rows)
print("Això NO és un criteri de validesa de la CI, però ajuda a veure biaix/skew del bootstrap:")
display(diag)

bias_summary = (
    results.groupby("coord")
    .agg(mean_abs_bias=("boot_bias", lambda s: np.nanmean(np.abs(s))),
         max_abs_bias=("boot_bias", lambda s: np.nanmax(np.abs(s))),
         mean_se=("boot_se", "mean"))
)
print("Biaix bootstrap per coordenada:")
display(bias_summary)


In [ ]:

# Floor automàtic: primer Delta on la banda inferior triada de T_env queda per sobre del null 95% pooled.
floor = infer_floor(results, coord_for_floor="T_env", require_ci=True)
print("Discrimination floor inferit:", floor)

plot_f2b(results, OUTDIR, floor=floor)
contrast = plot_f3_contrast(prep, OUTDIR, floor=floor, annotate="median")

print("Figures desades a:")
print(OUTDIR / "F2b_scale_std_fixed.png")
print(OUTDIR / "F2b_scale_std_fixed.pdf")
print(OUTDIR / "F3_intrawalk_vs_threshold_fixed.png")
print(OUTDIR / "F3_intrawalk_vs_threshold_fixed.pdf")

# Mostrar les figures dins el notebook
from IPython.display import Image, display
for p in [OUTDIR / "F2b_scale_std_fixed.png", OUTDIR / "F3_intrawalk_vs_threshold_fixed.png"]:
    display(Image(filename=str(p)))


In [ ]:

# Taula resum per escriure al TFM.
summary = (
    results.pivot_table(index="delta", columns="coord", values="obs")
    .rename(columns={"T_corr": "eps2_Tcorr", "HDX_corr": "eps2_HDX", "T_env": "eps2_Tenv"})
)
summary["Tenv_over_Tcorr"] = summary["eps2_Tenv"] / summary["eps2_Tcorr"].replace(0, np.nan)
summary["Tenv_over_HDX"] = summary["eps2_Tenv"] / summary["eps2_HDX"].replace(0, np.nan)

display(summary.round(4))
summary.to_csv(OUTDIR / "F2b_summary_ratios.csv")



## Mitjana o mediana?

Per aquesta figura jo mantindria la **mediana** com a agregador principal perquè la pregunta és “quina separabilitat local típica tinc a una finestra de mida \(\Delta\)?”. La mitjana pot quedar dominada per finestres d'extrem o per centres amb composició rara de classes.

Si el tribunal et demana una magnitud tipus “esperança sobre totes les finestres”, torna a executar amb:

```python
AGGREGATOR = "weighted_mean"
```

Això fa una mitjana ponderada pel nombre de punts de cada finestra. Jo la posaria com a robustesa, no com a figura principal.


In [ ]:

# Opcional: si tens el _f2b_data.csv antic a la mateixa carpeta, això mostra el problema de les bandes antigues.
old_csv = Path("_f2b_data.csv")
if old_csv.exists():
    old = pd.read_csv(old_csv)
    old["covered_old_percentile"] = (old["lo"] <= old["obs"]) & (old["obs"] <= old["hi"])
    print("Punts antics on l'observat queda fora del band percentil:")
    display(old.loc[~old["covered_old_percentile"], ["coord", "delta", "obs", "lo", "hi", "null95"]])
else:
    print("No hi ha _f2b_data.csv antic en aquesta carpeta; saltem aquest audit.")
